🎯 Goal of Notebook 04

Convert raw telemetry into features that help the model detect:

Abnormal machine behavior

before failures happen.

In [183]:
import numpy as numpy
import pandas as pd

In [184]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"

print(DATA_DIR)

f:\Projects\Advanced-Predictive-Maintenance\data\raw


In [185]:
import pandas as pd

errors = pd.read_csv(DATA_DIR / "PdM_errors.csv")
telemetry = pd.read_csv(DATA_DIR / "PdM_telemetry.csv")
machines = pd.read_csv(DATA_DIR / "PdM_machines.csv")
maint = pd.read_csv(DATA_DIR / "PdM_maint.csv")
failures = pd.read_csv(DATA_DIR / "PdM_failures.csv")

In [186]:
telemetry["datetime"] = pd.to_datetime(telemetry["datetime"])

master_df = telemetry.merge(
    machines,
    on="machineID",
    how="left"
)

master_df = master_df.sort_values(
    ["machineID", "datetime"]
)

master_df.head()

,datetime,machineID,volt,rotate,pressure,vibration,model,age
0,2015-01-01 06:00:00,1,176.217853,418.504078,113.077935,45.087686,model3,18
1,2015-01-01 07:00:00,1,162.879223,402.747490,95.460525,43.413973,model3,18
2,2015-01-01 08:00:00,1,170.989902,527.349825,75.237905,34.178847,model3,18
3,2015-01-01 09:00:00,1,162.462833,346.149335,109.248561,41.122144,model3,18
4,2015-01-01 10:00:00,1,157.610021,435.376873,111.886648,25.990511,model3,18


Step 2: Rolling Features
Rolling Mean (24 Hours)

For vibration:

In [187]:
master_df["vibration_mean_24h"] = (
    master_df.groupby("machineID")["vibration"]
    .transform(
        lambda x: x.rolling(
            window=24,
            min_periods=1
        ).mean()
    )
)

In [188]:
sensors = [
    "volt",
    "rotate",
    "pressure",
    "vibration"
]

for col in sensors:

    master_df[f"{col}_mean_24h"] = (
        master_df.groupby("machineID")[col]
        .transform(
            lambda x: x.rolling(
                24,
                min_periods=1
            ).mean()
        )
    )

Rolling Standard Deviation

In [189]:
for col in sensors:

    master_df[f"{col}_std_24h"] = (
        master_df.groupby("machineID")[col]
        .transform(
            lambda x: x.rolling(
                24,
                min_periods=1
            ).std()
        )
    )

1 Hour Lag

In [190]:
for col in sensors:

    master_df[f"{col}_lag1"] = (
        master_df.groupby("machineID")[col]
        .shift(1)
    )

6 Hour Lag

In [191]:
for col in sensors:

    master_df[f"{col}_lag6"] = (
        master_df.groupby("machineID")[col]
        .shift(6)
    )

24 Hour Lag

In [192]:
for col in sensors:

    master_df[f"{col}_lag24"] = (
        master_df.groupby("machineID")[col]
        .shift(24)
    )

Step 5: Trend Features
Why?

We found rotation decreases before failures.

Trend captures:

Current - Previous

Rate of Change

In [193]:
for col in sensors:

    master_df[f"{col}_change"] = (
        master_df[col]
        -
        master_df[f"{col}_lag1"]
    )

Percentage Change

In [194]:
for col in sensors:

    master_df[f"{col}_pct_change"] = (
        master_df.groupby("machineID")[col]
        .pct_change()
    )

Maintenance Features

In [195]:
maint.head()

,datetime,machineID,comp
0,2014-06-01 06:00:00,1,comp2
1,2014-07-16 06:00:00,1,comp4
2,2014-07-31 06:00:00,1,comp3
3,2014-12-13 06:00:00,1,comp1
4,2015-01-05 06:00:00,1,comp4


In [196]:
maint["datetime"] = pd.to_datetime(
    maint["datetime"]
)

maint = maint.sort_values(
    ["machineID","datetime"]
)

In [197]:
master_df = pd.merge_asof(
    master_df.sort_values("datetime"),
    maint.sort_values("datetime"),
    on="datetime",
    by="machineID",
    direction="backward"
)

Rename:

In [198]:
maint_feature = maint.copy()

In [199]:
maint_feature = maint_feature.rename(
    columns={
        "datetime": "maintenance_time"
    }
)

In [200]:
maint_feature = maint_feature.sort_values(
    ["machineID", "maintenance_time"]
)

In [201]:
master_df = master_df.sort_values(
    ["machineID", "datetime"]
)

This is a powerful feature.

In [202]:
master_df.shape

(876100, 37)

In [203]:
master_df.head()

,datetime,machineID,volt,rotate,pressure,vibration,model,age,vibration_mean_24h,volt_mean_24h,...,vibration_lag24,volt_change,rotate_change,pressure_change,vibration_change,volt_pct_change,rotate_pct_change,pressure_pct_change,vibration_pct_change,comp
0,2015-01-01 06:00:00,1,176.217853,418.504078,113.077935,45.087686,model3,18,45.087686,176.217853,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,comp1
168,2015-01-01 07:00:00,1,162.879223,402.747490,95.460525,43.413973,model3,18,44.250829,169.548538,...,NaN,-13.338630,-15.756589,-17.617410,-1.673713,-0.075694,-0.037650,-0.155799,-0.037121,comp1
241,2015-01-01 08:00:00,1,170.989902,527.349825,75.237905,34.178847,model3,18,40.893502,170.028993,...,NaN,8.110680,124.602336,-20.222621,-9.235126,0.049796,0.309381,-0.211843,-0.212722,comp1
398,2015-01-01 09:00:00,1,162.462833,346.149335,109.248561,41.122144,model3,18,40.950662,168.137453,...,NaN,-8.527069,-181.200490,34.010656,6.943297,-0.049869,-0.343606,0.452042,0.203146,comp1
419,2015-01-01 10:00:00,1,157.610021,435.376873,111.886648,25.990511,model3,18,37.958632,166.031967,...,NaN,-4.852812,89.227538,2.638087,-15.131633,-0.029870,0.257772,0.024148,-0.367968,comp1


Droping the component Column

In [204]:
master_df = master_df.drop(
    columns=["comp"],
    errors="ignore"
)

Drop Data Column because They don't capture patterns

In [205]:
#master_df = master_df.drop(
 #   columns=["datetime"]
#)

Handle model

In [206]:
master_df = pd.get_dummies(
    master_df,
    columns=["model"],
    drop_first=True
)

In [207]:
master_df.head()

,datetime,machineID,volt,rotate,pressure,vibration,age,vibration_mean_24h,volt_mean_24h,rotate_mean_24h,...,rotate_change,pressure_change,vibration_change,volt_pct_change,rotate_pct_change,pressure_pct_change,vibration_pct_change,model_model2,model_model3,model_model4
0,2015-01-01 06:00:00,1,176.217853,418.504078,113.077935,45.087686,18,45.087686,176.217853,418.504078,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,True,False
168,2015-01-01 07:00:00,1,162.879223,402.747490,95.460525,43.413973,18,44.250829,169.548538,410.625784,...,-15.756589,-17.617410,-1.673713,-0.075694,-0.037650,-0.155799,-0.037121,False,True,False
241,2015-01-01 08:00:00,1,170.989902,527.349825,75.237905,34.178847,18,40.893502,170.028993,449.533798,...,124.602336,-20.222621,-9.235126,0.049796,0.309381,-0.211843,-0.212722,False,True,False
398,2015-01-01 09:00:00,1,162.462833,346.149335,109.248561,41.122144,18,40.950662,168.137453,423.687682,...,-181.200490,34.010656,6.943297,-0.049869,-0.343606,0.452042,0.203146,False,True,False
419,2015-01-01 10:00:00,1,157.610021,435.376873,111.886648,25.990511,18,37.958632,166.031967,426.025520,...,89.227538,2.638087,-15.131633,-0.029870,0.257772,0.024148,-0.367968,False,True,False


Handle NAN's

In [208]:
master_df.isnull().sum().sort_values(
    ascending=False
).head(20)

pressure_lag24          2400
vibration_lag24         2400
rotate_lag24            2400
volt_lag24              2400
vibration_lag6           600
pressure_lag6            600
rotate_lag6              600
volt_lag6                600
vibration_change         100
rotate_lag1              100
vibration_pct_change     100
pressure_pct_change      100
rotate_pct_change        100
volt_change              100
vibration_lag1           100
pressure_change          100
pressure_lag1            100
volt_lag1                100
vibration_std_24h        100
volt_pct_change          100
dtype: int64

In [209]:
master_df = master_df.fillna(0)

Check Infinite Values

In [210]:
import numpy as np

np.isinf(master_df.select_dtypes(
    include=np.number
)).sum()

machineID               0
volt                    0
rotate                  0
pressure                0
vibration               0
age                     0
vibration_mean_24h      0
volt_mean_24h           0
rotate_mean_24h         0
pressure_mean_24h       0
volt_std_24h            0
rotate_std_24h          0
pressure_std_24h        0
vibration_std_24h       0
volt_lag1               0
rotate_lag1             0
pressure_lag1           0
vibration_lag1          0
volt_lag6               0
rotate_lag6             0
pressure_lag6           0
vibration_lag6          0
volt_lag24              0
rotate_lag24            0
pressure_lag24          0
vibration_lag24         0
volt_change             0
rotate_change           0
pressure_change         0
vibration_change        0
volt_pct_change         0
rotate_pct_change       0
pressure_pct_change     0
vibration_pct_change    0
dtype: int64

Before Training Run These Checks
Missing Values , 
Infinite Values ,
Data Types


In [211]:
print(master_df.isnull().sum().sum())
print(np.isinf(master_df.select_dtypes(include=np.number)).sum().sum())
print(master_df.dtypes.value_counts())

0
0
float64           32
bool               3
int64              2
datetime64[us]     1
Name: count, dtype: int64


In [212]:
feature_df = master_df.select_dtypes(
    include=["int64", "float64", "bool"]
)

In [213]:
feature_df = feature_df.drop(
    columns=["machineID"]
)

In [214]:
feature_df.columns

Index(['volt', 'rotate', 'pressure', 'vibration', 'age', 'vibration_mean_24h',
       'volt_mean_24h', 'rotate_mean_24h', 'pressure_mean_24h', 'volt_std_24h',
       'rotate_std_24h', 'pressure_std_24h', 'vibration_std_24h', 'volt_lag1',
       'rotate_lag1', 'pressure_lag1', 'vibration_lag1', 'volt_lag6',
       'rotate_lag6', 'pressure_lag6', 'vibration_lag6', 'volt_lag24',
       'rotate_lag24', 'pressure_lag24', 'vibration_lag24', 'volt_change',
       'rotate_change', 'pressure_change', 'vibration_change',
       'volt_pct_change', 'rotate_pct_change', 'pressure_pct_change',
       'vibration_pct_change', 'model_model2', 'model_model3', 'model_model4'],
      dtype='str')

In [215]:
feature_df.shape

(876100, 36)

In [216]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

In [217]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(
    feature_df
)

In [218]:
model = IsolationForest(
    contamination=0.01,
    random_state=42,
    n_estimators=100
)

model.fit(X_scaled)

,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",100
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.01
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary ` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",42
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary `... versionadded:: 0.21",False


In [219]:
master_df["anomaly_score"] = (
    model.decision_function(X_scaled)
)

master_df["anomaly_label"] = (
    model.predict(X_scaled)
)

In [220]:
master_df["anomaly_label"].value_counts()

anomaly_label
 1    867339
-1      8761
Name: count, dtype: int64

In [221]:
master_df.isnull().sum().sum()

0

In [222]:
master_df.dtypes.value_counts()

float64           33
bool               3
int64              2
datetime64[us]     1
int32              1
Name: count, dtype: int64

In [223]:
master_df["anomaly_score"].describe()

count    876100.000000
mean          0.083194
std           0.028342
min          -0.152387
25%           0.067367
50%           0.087658
75%           0.103613
max           0.152949
Name: anomaly_score, dtype: float64

🎯 Goal

We want to answer:

When a machine actually failed, did our system raise an alert before the failure?

Example:

Failure Time:
2025-03-10 10:00

Alert Time:
2025-03-10 03:00

Difference:
7 Hours

✅ Success

In [224]:
#create anomaly dataset
master_df["anomaly_label"]

0        -1
168      -1
241      -1
398      -1
419      -1
         ..
875611    1
875702    1
875806    1
875928    1
876099    1
Name: anomaly_label, Length: 876100, dtype: int32

In [225]:
anomalies = master_df[
    master_df["anomaly_label"] == -1
].copy()

print(anomalies.shape)

(8761, 40)


In [226]:
master_df["datetime"] = pd.to_datetime(
    master_df["datetime"]
)

failures["datetime"] = pd.to_datetime(
    failures["datetime"]
)

anomalies["datetime"] = pd.to_datetime(
    anomalies["datetime"]
)

In [227]:
total_failures = len(failures)

alerts_before_failure = 0

alert_lead_times = []

In [228]:
for _, failure in failures.iterrows():

    machine_id = failure["machineID"]

    failure_time = failure["datetime"]

    machine_alerts = anomalies[
        (anomalies["machineID"] == machine_id)
        &
        (
            anomalies["datetime"]
            >= failure_time - pd.Timedelta(hours=24)
        )
        &
        (
            anomalies["datetime"]
            < failure_time
        )
    ]

    if len(machine_alerts) > 0:

        alerts_before_failure += 1

        latest_alert = machine_alerts[
            "datetime"
        ].max()

        lead_time = (
            failure_time
            -
            latest_alert
        ).total_seconds() / 3600

        alert_lead_times.append(
            lead_time
        )

In [232]:
detection_rate = (
    alerts_before_failure
    /
    total_failures
) * 100

print(
    f"Failures: {total_failures}"
)

print(
    f"Detected: {alerts_before_failure}"
)

print(
    f"Detection Rate: {detection_rate:.2f}%"
)

Failures: 761
Detected: 421
Detection Rate: 55.32%


In [230]:
import numpy as np

avg_lead_time = np.mean(
    alert_lead_times
)

print(
    f"Average Lead Time: {avg_lead_time:.2f} hours"
)

Average Lead Time: 7.64 hours


In [233]:
#Save Model
import joblib
import os

os.makedirs("models", exist_ok=True)

joblib.dump(
    model,
    "models/isolation_forest.pkl"
)

print("Model Saved")

Model Saved


Save Scaler

Very important.

Your model was trained on:

In [235]:
X_scaled = scaler.fit_transform(feature_df)

In [236]:
joblib.dump(
    scaler,
    "models/scaler.pkl"
)

print("Scaler Saved")

Scaler Saved


Save Feature List
new_data.columns

doesn't match training columns.

Predictions will fail.

In [237]:
feature_columns = feature_df.columns.tolist()

joblib.dump(
    feature_columns,
    "models/features.pkl"
)

['models/features.pkl']

In [238]:
#Test Loading
loaded_model = joblib.load(
    "models/isolation_forest.pkl"
)

loaded_scaler = joblib.load(
    "models/scaler.pkl"
)

loaded_features = joblib.load(
    "models/features.pkl"
)